# BEAVER-ES: Time Series
***

***Author**: Chus Casado Rodríguez*<br>
***Date**:01-09-2026*<br>

**Introduction:**<br>



**Outputs:**<br>


**To do**:<br>
* [] Filter stations with wrong catchment polygon and fix it.
* [x] Probably the timestamps in CERRA are shifted one day (like EMO1).
* [x] How to trim the meteo time series at the start? Should I include one extra year as initial condition for the first discharge observation?

In [161]:
from pathlib import Path
from tqdm.auto import tqdm
import json
import logging
logger = logging.getLogger(__name__)

import pandas as pd
import geopandas as gpd
from sklearn.model_selection import train_test_split

from ocab.config import Config
import ocab.variables as VARS
from ocab.timeseries.utils import time_encoding
from ocab.utils.sampling import create_sample_file, create_period_file

In [ ]:
def valid_timeseries(
        answers: pd.DataFrame, 
        column: str ='incorrect_ts', 
        inplace: bool = False
    ) -> pd.DataFrame | None:
    """Creates a DataFrame with fields of valid time series, based on the answers to the 
    online questionnaire.
    
    Parameters
    ----------
    answers: pd.DataFrame
        The raw answers to the online questionnaire.
    columnn: str
        Name of the column indicating the incorrect time series.
    inplace: bool
        If True, the original DataFrame is modified in place. If False, a new DataFrame is 
        returned with the valid time series fields.

    Returns
    -------
    pd.DataFrame or None
        * pd.DataFrame : Binary indicator DataFrame if `inplace=False`.
        * None : If `inplace=True`.
    """

    # split list of variables
    incorrect_ts = answers[column].str.split(', ').copy()

    # extract possible variables and create a rename mapping
    old_vars = incorrect_ts.explode().dropna().unique().tolist()
    rename_vars = {var: var.split()[0].lower() for var in old_vars}
    new_vars = list(rename_vars.values())

    # apply rename mapping
    incorrect_ts = incorrect_ts.apply(
        lambda lst: [rename_vars[x] for x in lst if x in rename_vars]
        if isinstance(lst, list) else []
    )

    # create fields of valid time series
    valid_ts = pd.DataFrame(1, index=answers.index, columns=new_vars, dtype=int)
    for ID, lst in incorrect_ts.items():
        if len(lst) > 0:
            valid_ts.loc[ID, lst] = 0

    if inplace:
        answers.drop(columns=[column], inplace=True)
        for col in valid_ts.columns:
            answers[col] = valid_ts[col]
    else:
        return valid_ts

In [143]:
def combine_periods(row, days: int = 365):
    starts = [row['start_1']]
    ends = [row['end_1']]
    if row['second_period'] == 'Yes':
        starts.append(row['start_2'])
        ends.append(row['end_2'])
        if row['third_period'] == 'Yes':
            starts.append(row['start_3'])
            ends.append(row['end_3'])
    starts = pd.to_datetime(starts, dayfirst=True) - pd.Timedelta(days=days)
    ends = pd.to_datetime(ends, dayfirst=True)
    return pd.Series({"start_dates": starts, "end_dates": ends})

## Configuration

In [162]:
cfg = Config('config_BEAVERS_v100.yml')

# point layer
filename = 'dams.geojson'

# paths
# path_in = cfg.path_dataset / 'preprocessing' / 'timeseries'
path_in = Path('../../docs/timeseries/reservoirs')

# questionnaire
url_form = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vQ-uWba6lG8gvXwSd5kMmU2lHp8hgPG4agc8LhikPO-jQdbWHZCeMYFbLtbo6jW-436SgRenZoxiaaH/pub?gid=1682822662&single=true&output=csv'

# import timeseries metadata
with open('metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

## Data

### Dams

In [3]:
# load points
points = gpd.read_file(cfg.path_gis / filename).set_index('id')
print(f'no. points: {len(points):4}')

# identify basins
basins = points['basin'].unique()
print(f'no. basins: {len(basins):4}')

no. points:  374
no. basins:   12


### Selection

I load and handle the answers to the questionnaire in the [website](https://casadoj.github.io/of_camels_and_beavers/).

In [144]:
# load answers to the online questionnaire
answers = pd.read_csv(url_form, parse_dates=True)

# rename columns
rename_cols = {
    'Marca temporal': 'timestamp', 
    'Reservoir ID': 'ID', 
    'Gauging stations directly upstream the reservoir.': 'upstream',
    'Gauging stations directly downstream the reservoir.': 'downstream',
    'Is the catchment polygon correct?': 'catchment',
    'Is any of these time series clearly incorrect?': 'incorrect_ts',
    'Start date (1st period)': 'start_1', 
    'End date (1st period)': 'end_1',
    'Is there a second period of high-quality data?': 'second_period',
    'Start date (2nd period)': 'start_2', 
    'End date (2nd period)': 'end_2',
    'Is there a third period of high-quality data?': 'third_period',
    'Start date (3rd period)': 'start_3', 
    'End date (3rd period)': 'end_3',
    'Is the indicated reservoir use correct?': 'use_correct',
    "What's the correct main use of the reservoir?": 'main_use',
    'Raise any other issue in the reservoir attributes or time series. E.g., filling values above 1 may indicate an error in the storage capacity.': 'comments',
    # 'email', 
}
answers.rename(columns=rename_cols, inplace=True)

# Check if all stations have been revised
for basin in basins:
    missing = points[points['basin'] == basin].index.difference(answers['ID'])
    if len(missing > 0):
        print('{0:<12}: {1}'.format(basin, list(missing)))

# keep only selected stations
answers = answers[answers['ID'].isin(points.index)]
print('Raw data')
print(f'No. answers:\t\t{len(answers)}')
print(f'No. unique stations:\t{len(answers["ID"].unique())}')


# # select stations with natural or semi-natural regimes
# # if multiple answers, I take the majority vote
# IDs = []
# for ID in answers['ID'].unique():
#     subset = answers[answers['ID'] == ID]
#     if len(subset) > 1:
#         if subset['regime'].value_counts().index[0] in ['Natural', 'Semi-natural']:
#             IDs.append(ID)
#     else:
#         if subset['regime'].item() in ['Natural', 'Semi-natural']:
#             IDs.append(ID)
# mask_id = answers['ID'].isin(IDs)
# mask_regime = answers['regime'].isin(['Natural', 'Semi-natural'])
# answers = answers[mask_id & mask_regime]

# print('\n(Semi)natural regime')
# print(f'No. answers:\t\t{len(answers)}')
# print(f'No. unique stations:\t{len(answers["ID"].unique())}')

# if duplicate stations, keep only the last answer
answers = answers.sort_values('timestamp').drop_duplicates('ID', keep='last')
answers = answers.set_index('ID', drop=True).sort_index(axis=0)


Raw data
No. answers:		384
No. unique stations:	374


In [145]:
# remove reservoirs with no valid time series
valid_ts = valid_timeseries(answers, inplace=False)
answers = answers[~(valid_ts == 0).all(axis=1)]

In [146]:
len(answers)

356

In [147]:
answers[variables] = 1

In [148]:

# combine selected periods
answers[['start_dates', 'end_dates']] = answers.apply(combine_periods, axis=1)

# drop some columns
drop = ['timestamp', 'start_1', 'end_1', 'second_period', 'start_2', 'end_2', 'third_period', 'start_3', 'end_3']
answers.drop(columns=drop, inplace=True)

## Export

### Samples and Periods

In [150]:
# define output folder
path_samples = cfg.path_dataset / 'selection'
path_samples.mkdir(exist_ok=True)
print(f'Samples will be saved in {path_samples}')

# keep points selected in the questionnaire
points = points.loc[points.index.intersection(answers.index)]

# only because SEGURA has few stations!!
points['stratify'] = points['basin']
# merge_group = points['basin'].isin(['JUCAR', 'SEGURA'])
# points.loc[merge_group, 'stratify'] = 'MEDITERRÁNEO'

# divide basins in train, validation and test sets
train_set, temp = train_test_split(
    points,
    train_size=cfg.train_size,
    random_state=cfg.seed,
    stratify=points['stratify']
)
val_set, test_set = train_test_split(
    temp,
    train_size=cfg.val_size / (1 - cfg.train_size),
    random_state=cfg.seed,
    stratify=temp['stratify']
)
print(f'train size:\t\t{len(train_set)}')
print(f'validation size:\t{len(val_set)}')
print(f'test size:\t\t{len(test_set)}')

# organize basin IDs according to the sample
samples = {
    'train': train_set.sort_index().index.to_list(),
    'validation': val_set.sort_index().index.to_list(),
    'test': test_set.sort_index().index.to_list(),
}

# create sample files
for name, sample in samples.items():

    # export TXT file

    # all dataset
    create_sample_file(cfg, sample, path_samples / f'basins_{name}.txt')

    # per basin
    for basin in basins:
        sample_basin = points[points['basin'] == basin].index.intersection(sample).tolist()
        if len(sample_basin) > 0:
            create_sample_file(cfg, sample_basin, path_samples / f'{basin.lower()}_{name}.txt')

    # export PKL file
    create_period_file(cfg, sample, answers, path_samples / f'periods_{name}.pkl')

Samples will be saved in /home/casadoj/Data/BEAVERS-ES/v1_0_0/selection
train size:		213
validation size:	71
test size:		72



### Time series

In [165]:
resops = pd.read_parquet(path_in / f'{ID}.parquet')

resops.head()


,storage_mcm,level_masl,outflow_cms,inflow_cms,filling,inflow_mm,outflow_mm,precip_mm_rocio,precip_max_mm_rocio,precip_std_mm_rocio,...,precip_std_mm_cerra,precip_frac_cerra,temp_degC_cerra,temp_min_degC_cerra,temp_max_degC_cerra,pet_mm_cerra,temp_dtr_degC_cerra,temp_degC_emo,pet_mm_emo,precip_mm_emo
date,,,,,,,,,,,,,,,,,,,,,
1980-10-01,0.30,197.139,0.09,NaN,0.056,NaN,0.250839,0.0000,0.00,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980-10-02,0.29,197.028,0.09,0.090,0.055,0.250839,0.250839,0.0075,0.01,0.00433,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980-10-03,0.29,196.997,0.02,NaN,0.055,NaN,0.055742,0.0000,0.00,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980-10-04,0.28,196.813,0.14,0.024,0.053,0.067613,0.390194,0.0000,0.00,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980-10-05,0.27,196.705,0.08,NaN,0.051,NaN,0.222968,0.0000,0.00,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [166]:
resops.shape

(16436, 28)

In [167]:
resops.dropna(axis=1, how='all').shape

(16436, 28)

In [154]:
cfg.path_records

PosixPath('/home/casadoj/Data/CEDEX/2020-2021')

In [155]:
cfg.path_timeseries

PosixPath('/home/casadoj/Data/BEAVERS-ES/v1_0_0/timeseries')

In [156]:
pd.read_parquet(f'../../docs/timeseries/reservoirs/{ID}.parquet')

,storage_mcm,level_masl,outflow_cms,inflow_cms,filling,inflow_mm,outflow_mm,precip_mm_rocio,precip_max_mm_rocio,precip_std_mm_rocio,...,precip_std_mm_cerra,precip_frac_cerra,temp_degC_cerra,temp_min_degC_cerra,temp_max_degC_cerra,pet_mm_cerra,temp_dtr_degC_cerra,temp_degC_emo,pet_mm_emo,precip_mm_emo
date,,,,,,,,,,,,,,,,,,,,,
1980-10-01,0.300,197.139,0.09,NaN,0.056,NaN,0.250839,0.0000,0.00,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980-10-02,0.290,197.028,0.09,0.090,0.055,0.250839,0.250839,0.0075,0.01,0.00433,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980-10-03,0.290,196.997,0.02,NaN,0.055,NaN,0.055742,0.0000,0.00,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980-10-04,0.280,196.813,0.14,0.024,0.053,0.067613,0.390194,0.0000,0.00,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980-10-05,0.270,196.705,0.08,NaN,0.051,NaN,0.222968,0.0000,0.00,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-26,1.897,208.726,0.00,NaN,0.357,NaN,0.000000,NaN,NaN,NaN,...,0.000016,0.0,14.651072,8.639837,22.378864,3.173647,13.739027,NaN,NaN,NaN
2025-09-27,1.891,208.696,0.00,NaN,0.355,NaN,0.000000,NaN,NaN,NaN,...,0.005575,0.0,17.269230,11.797565,22.557995,2.924145,10.760430,NaN,NaN,NaN
2025-09-28,1.885,208.667,0.00,NaN,0.354,NaN,0.000000,NaN,NaN,NaN,...,0.034290,0.0,18.225304,16.933422,20.979242,1.866819,4.045820,NaN,NaN,NaN


In [157]:
path_in

PosixPath('/home/casadoj/Data/BEAVERS-ES/v1_0_0/preprocessing/timeseries')

In [168]:
# define output folder
path_csv = cfg.path_timeseries / 'csv' #/ cfg.prefix
path_nc = cfg.path_timeseries / 'netcdf' #/ cfg.prefix
for path in [path_csv, path_nc]:
    path.mkdir(parents=True, exist_ok=True)
print(f'Time series will be saved in {cfg.path_timeseries}')

# process timeseries for each station
for ID in tqdm(answers.index, desc='points'):
    
    # TIME SERIES
    # ...........
    try:
        ts = pd.read_parquet(path_in / f'{ID}.parquet')
        ts.dropna(axis=1, how='all', inplace=True)
    except Exception as e:
        logger.error(f'Loading discharge timeseries for station {ID:04d}: {e}')
        continue
    
    # define time period
    start = min(answers.loc[ID, 'start_dates'])
    end = max(answers.loc[ID, 'end_dates'])
    ts = ts.loc[start:end]

    # TEMPORAL ENCODERS
    # .................
    ts['year'] = ts.index.year
    ts['month'] = ts.index.month
    ts['month_sin'], ts['month_cos'] = time_encoding(ts['month'], period=12)
    ts['weekofyear'] = ts.index.isocalendar().week.astype('uint32')
    ts['woy_sin'], ts['woy_cos'] = time_encoding(ts['weekofyear'], period=52)
    ts['dayofyear'] = ts.index.dayofyear
    ts['doy_sin'], ts['doy_cos'] = time_encoding(ts['dayofyear'], period=365)
    ts['dayofweek'] = ts.index.isocalendar().day.astype('uint32')
    ts['dow_sin'], ts['dow_cos'] = time_encoding(ts['dayofweek'], period=7)

    # EXPORT
    # ......
    
    # export CSV file
    ts.to_csv(path_csv / f'{cfg.prefix}_{ID}.csv', index=True)

    # export NetCDF file
    ds = ts.to_xarray()
    ds.attrs['Timezone'] = metadata['Timezone']
    ds.attrs['Sources'] = metadata['Sources']
    for var in ds.data_vars:
        var_short = '_'.join(var.split('_')[:2])  # remove dataset suffix
        if var_short in metadata['variables']:
            ds[var].attrs['long_name'] = metadata['variables'][var_short]['long_name']
            ds[var].attrs['units'] = metadata['variables'][var_short]['units']
    ts.to_xarray().to_netcdf(path_nc / f'{cfg.prefix}_{ID}.nc')

Time series will be saved in /home/casadoj/Data/BEAVERS-ES/v1_0_0/timeseries


points:   0%|          | 0/356 [00:00<?, ?it/s]